In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

def bt_american(call, underlying, strike, ttm, rf, b, ivol, N):
    """
    Binomial tree for American options following Julia implementation
    """
    dt = ttm / N
    u = np.exp(ivol * np.sqrt(dt))
    d = 1 / u
    pu = (np.exp(b * dt) - d) / (u - d)
    pd = 1.0 - pu
    df = np.exp(-rf * dt)
    z = 1 if call else -1

    # Build price tree at maturity
    prices = np.array([underlying * (u ** (N - i)) * (d ** i) for i in range(N + 1)])

    # Initialize option values at maturity
    values = np.maximum(z * (prices - strike), 0)

    # Backward induction
    for step in range(N - 1, -1, -1):
        for i in range(step + 1):
            price = underlying * (u ** (step - i)) * (d ** i)
            # Continuation value
            values[i] = df * (pu * values[i] + pd * values[i + 1])
            # Early exercise value
            exercise = max(0, z * (price - strike))
            values[i] = max(values[i], exercise)

    return values[0]

def american_option_greeks(S, K, T, r, q, sigma, option_type):
    """
    Calculate American option price and Greeks
    """
    call = (option_type == 'Call')
    b = r - q
    N = 500

    price = bt_american(call, S, K, T, r, b, sigma, N)

    # Delta - sensitivity to underlying price
    dS = 1.0
    price_up = bt_american(call, S + dS, K, T, r, b, sigma, N)
    price_down = bt_american(call, S - dS, K, T, r, b, sigma, N)
    delta = (price_up - price_down) / (2 * dS)

    # Gamma - second derivative with respect to price
    gamma = (price_up - 2 * price + price_down) / (dS ** 2)

    # Vega - sensitivity to volatility
    dv = 0.01
    price_vol_up = bt_american(call, S, K, T, r, b, sigma + dv, N)
    vega = (price_vol_up - price) / dv

    # Rho - sensitivity to risk-free rate (only rf changes, b stays same)
    dr = 0.001
    price_r_up = bt_american(call, S, K, T, r + dr, b, sigma, N)
    rho = (price_r_up - price) / dr

    # Theta - time decay (positive time direction)
    dt = 1 / 365
    price_t_up = bt_american(call, S, K, T + dt, r, b, sigma, N)
    theta = (price_t_up - price) / dt

    return price, delta, gamma, vega, rho, theta

# Read data
DATA_DIR = Path.cwd() / "testfiles_" / "data"
CSV_PATH = DATA_DIR / "test12_1.csv"
df = pd.read_csv(CSV_PATH, header=0)

df = df.dropna(subset=['ID'])

results = []
for _, row in df.iterrows():
    S = row['Underlying']
    K = row['Strike']
    T = row['DaysToMaturity'] / row['DayPerYear']
    r = row['RiskFreeRate']
    q = row['DividendRate']
    sigma = row['ImpliedVol']
    option_type = row['Option Type']

    value, delta, gamma, vega, rho, theta = american_option_greeks(
        S, K, T, r, q, sigma, option_type
    )

    results.append({
        'ID': int(row['ID']),
        'Value': value,
        'Delta': delta,
        'Gamma': gamma,
        'Vega': vega,
        'Rho': rho,
        'Theta': theta
    })

output_df = pd.DataFrame(results)
print(output_df)

   ID      Value     Delta     Gamma       Vega        Rho      Theta
0   1   3.259347  0.547643  0.059244  14.653661  -0.446455  12.960975
1   2   2.692775 -0.463483  0.060507  14.615371  -0.253688   8.887079
2   3  22.050266  0.680852  0.008986  35.904557 -22.792528   7.275168
3   4  21.019233 -0.490073  0.002592  40.744104 -14.083944   5.951540
